In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/TrustLens/CIFAKE.zip" # Corrected path
EXTRACT_PATH = "/content/CIFAKE"

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted!")
print(os.listdir(EXTRACT_PATH))

Dataset extracted!
['train', 'test']


In [ ]:
import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from torchvision.models import EfficientNet_B0_Weights

In [ ]:
DATA_DIR = "/content/CIFAKE"

TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Separate dataset objects allow different transforms
full_train_aug = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
full_train_eval = datasets.ImageFolder(TRAIN_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

print("Classes:", full_train_aug.class_to_idx)
print("Training images:", len(full_train_aug))
print("Test images:", len(test_dataset))

Classes: {'FAKE': 0, 'REAL': 1}
Training images: 100000
Test images: 20000


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

targets = np.array(full_train_aug.targets)
indices = np.arange(len(targets))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.10,
    random_state=SEED,
    stratify=targets
)

train_dataset = Subset(full_train_aug, train_idx)
val_dataset = Subset(full_train_eval, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 90000
Validation: 10000
Test: 20000


In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT

model = models.efficientnet_b0(weights=weights)

# Freeze the feature extractor initially
for param in model.features.parameters():
    param.requires_grad = False

# Replace final classifier
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(in_features, 2)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print(model.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 159MB/s]


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=2, bias=True)
)


In [ ]:
EPOCHS = 10

best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):

    # ---------------- TRAINING ----------------
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = running_loss / train_total
    train_acc = 100 * train_correct / train_total

    # ---------------- VALIDATION ----------------
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = 100 * val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.2f}%"
    )

    # Save best validation model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

        torch.save(
            best_model_wts,
            "/content/ai_detector_best.pth"
        )

print("Best validation accuracy:", best_val_acc)

Epoch [1/10] Train Loss: 0.3541 Train Acc: 84.55% Val Loss: 0.2827 Val Acc: 88.16%
Epoch [2/10] Train Loss: 0.3405 Train Acc: 85.11% Val Loss: 0.2771 Val Acc: 88.30%
Epoch [3/10] Train Loss: 0.3397 Train Acc: 85.33% Val Loss: 0.2795 Val Acc: 88.11%
Epoch [4/10] Train Loss: 0.3432 Train Acc: 85.23% Val Loss: 0.2794 Val Acc: 87.89%
Epoch [5/10] Train Loss: 0.3393 Train Acc: 85.36% Val Loss: 0.2842 Val Acc: 88.13%
Epoch [6/10] Train Loss: 0.3435 Train Acc: 85.18% Val Loss: 0.2806 Val Acc: 88.16%
Epoch [7/10] Train Loss: 0.3404 Train Acc: 85.44% Val Loss: 0.2938 Val Acc: 87.45%
Epoch [8/10] Train Loss: 0.3410 Train Acc: 85.32% Val Loss: 0.2866 Val Acc: 87.86%
Epoch [9/10] Train Loss: 0.3408 Train Acc: 85.44% Val Loss: 0.2764 Val Acc: 88.39%
Epoch [10/10] Train Loss: 0.3438 Train Acc: 85.20% Val Loss: 0.2779 Val Acc: 88.12%
Best validation accuracy: 88.39


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

model.load_state_dict(
    torch.load("/content/ai_detector_best.pth", map_location=device)
)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

test_acc = accuracy_score(all_labels, all_preds)

print(f"Test Accuracy: {test_acc * 100:.2f}%")

print(classification_report(
    all_labels,
    all_preds,
    target_names=test_dataset.classes,
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Test Accuracy: 88.79%
              precision    recall  f1-score   support

        FAKE     0.8838    0.8933    0.8885     10000
        REAL     0.8921    0.8825    0.8873     10000

    accuracy                         0.8879     20000
   macro avg     0.8879    0.8879    0.8879     20000
weighted avg     0.8879    0.8879    0.8879     20000

Confusion Matrix:
[[8933 1067]
 [1175 8825]]


In [ ]:
from PIL import Image

def predict_image(image_path, model):
    image = Image.open(image_path).convert("RGB")
    image = eval_transform(image).unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():
        outputs = model(image)
        probabilities = torch.softmax(outputs, dim=1)[0]

    predicted_idx = probabilities.argmax().item()
    predicted_class = test_dataset.classes[predicted_idx]
    confidence = probabilities[predicted_idx].item() * 100

    return predicted_class, confidence


# Replace with the path to your image
image_path = "/content/Corn.jfif"

prediction, confidence = predict_image(image_path, model)

print("Prediction:", prediction)
print(f"Confidence: {confidence:.2f}%")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Corn.jfif'

In [ ]:
import shutil

DRIVE_OUTPUT = "/content/drive/MyDrive/TrustLens/ai_detector_best.pth"

os.makedirs(os.path.dirname(DRIVE_OUTPUT), exist_ok=True)

shutil.copy(
    "/content/ai_detector_best.pth",
    DRIVE_OUTPUT
)

print("Model saved to:", DRIVE_OUTPUT)

Model saved to: /content/drive/MyDrive/TrustLens/ai_detector_best.pth


In [ ]:
from google.colab import files

files.download("/content/ai_detector_best.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>